In [1]:
import trimesh
import numpy as np
from pathlib import Path
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

FRAME_CHECK_DIR = Path("frame_check")
ply_files = sorted(FRAME_CHECK_DIR.rglob("*.ply"))
print(f"Found {len(ply_files)} .ply files:")
for f in ply_files:
    print(f"  {f.relative_to(FRAME_CHECK_DIR)}")

Found 542 .ply files:
  0bda85e448ac4b3ea30fb63d02c97f67/az000_gt.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az000_gt_aligned.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az000_triposr.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az036_gt.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az036_gt_aligned.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az036_triposr.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az072_gt.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az072_gt_aligned.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az072_triposr.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az108_gt.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az108_gt_aligned.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az108_triposr.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az144_gt.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az144_gt_aligned.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az144_triposr.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az180_gt.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az180_gt_aligned.ply
  0bda85e448ac4b3ea30fb63d02c97f67/az180_triposr.ply
  0bda85e448ac4b3ea30fb63d02c97f67/a

In [2]:
MAX_FACES = 5000

def render_mesh_on_ax(mesh, ax, color='steelblue', alpha=0.6, label='', auto_lim=False):
    verts = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    if len(faces) > MAX_FACES:
        idx = np.random.choice(len(faces), MAX_FACES, replace=False)
        faces = faces[idx]
    polys = verts[faces]
    ax.add_collection3d(Poly3DCollection(polys, alpha=alpha,
                                         facecolor=color, edgecolor='none'))
    if auto_lim:
        lo, hi = verts.min(axis=0), verts.max(axis=0)
        ctr = (lo + hi) / 2
        half = (hi - lo).max() / 2 * 1.15
        ax.set_xlim(ctr[0] - half, ctr[0] + half)
        ax.set_ylim(ctr[1] - half, ctr[1] + half)
        ax.set_zlim(ctr[2] - half, ctr[2] + half)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    if label:
        ax.set_title(label, fontsize=9)

def set_shared_lim(ax, meshes):
    all_verts = np.concatenate([np.asarray(m.vertices) for m in meshes if m is not None])
    lo, hi = all_verts.min(axis=0), all_verts.max(axis=0)
    ctr = (lo + hi) / 2
    half = (hi - lo).max() / 2 * 1.15
    ax.set_xlim(ctr[0] - half, ctr[0] + half)
    ax.set_ylim(ctr[1] - half, ctr[1] + half)
    ax.set_zlim(ctr[2] - half, ctr[2] + half)

def icp_rigid(source_pts, target_mesh, max_iter=80):
    """ICP via SVD: find rotation + translation aligning source points to target mesh surface."""
    pts = source_pts.copy()
    T_total = np.eye(4)
    prev_cost = float('inf')

    for _ in range(max_iter):
        closest, dists, _ = trimesh.proximity.closest_point(target_mesh, pts)
        cost = float(dists.mean())
        if abs(prev_cost - cost) < 1e-8:
            break
        prev_cost = cost

        src_c = pts.mean(axis=0)
        tgt_c = closest.mean(axis=0)
        H = (pts - src_c).T @ (closest - tgt_c)
        U, _, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[-1] *= -1
            R = Vt.T @ U.T
        t = tgt_c - R @ src_c

        pts = (R @ pts.T).T + t
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = t
        T_total = T @ T_total

    return T_total, cost

def align_gt_to_triposr(gt_mesh, tri_mesh, n_samples=5000):
    """Find best scale + rotation + translation via bbox pre-align then ICP."""
    gt_center = (gt_mesh.bounds[0] + gt_mesh.bounds[1]) / 2
    tri_center = (tri_mesh.bounds[0] + tri_mesh.bounds[1]) / 2
    gt_ext = gt_mesh.bounds[1] - gt_mesh.bounds[0]
    tri_ext = tri_mesh.bounds[1] - tri_mesh.bounds[0]
    scale = float(np.mean(tri_ext / (gt_ext + 1e-8)))

    aligned = gt_mesh.copy()
    aligned.apply_translation(-gt_center)
    aligned.apply_scale(scale)
    aligned.apply_translation(tri_center)

    src_pts = np.array(aligned.sample(n_samples))
    matrix, cost = icp_rigid(src_pts, tri_mesh)
    aligned.apply_transform(matrix)

    return aligned, scale, cost

In [3]:
# Build pairs: group by sample key -> {triposr: path, gt: path}
pairs = {}
for f in ply_files:
    name = f.stem
    if '_triposr' in name:
        key = name.replace('_triposr', '')
        pairs.setdefault(key, {})['triposr'] = f
    elif '_gt' in name:
        key = name.replace('_gt_rotated', '').replace('_gt', '')
        pairs.setdefault(key, {})['gt'] = f

sorted_keys = sorted(pairs.keys())
print(f"{len(sorted_keys)} pairs found")
for k in sorted_keys:
    has = ', '.join(pairs[k].keys())
    print(f"  {k}: {has}")

21 pairs found
  0bda85e448ac4b3ea30fb63d02c97f67_az036: gt, triposr
  az000: gt, triposr
  az000_aligned: gt
  az036: gt, triposr
  az036_aligned: gt
  az072: gt, triposr
  az072_aligned: gt
  az108: gt, triposr
  az108_aligned: gt
  az144: gt, triposr
  az144_aligned: gt
  az180: gt, triposr
  az180_aligned: gt
  az216: gt, triposr
  az216_aligned: gt
  az252: gt, triposr
  az252_aligned: gt
  az288: gt, triposr
  az288_aligned: gt
  az324: gt, triposr
  az324_aligned: gt


In [ ]:
# Render ALL pairs: 4 columns per row (GT | TripoSR | Raw Overlay | Aligned Overlay)
NCOLS = 4
n = len(sorted_keys)
if n == 0:
    print("No pairs found.")
else:
    fig = plt.figure(figsize=(5 * NCOLS, 4.5 * n))
    for row, key in enumerate(sorted_keys):
        p = pairs[key]
        gt_mesh = trimesh.load(str(p['gt'])) if 'gt' in p else None
        tri_mesh = trimesh.load(str(p['triposr'])) if 'triposr' in p else None

        # Col 1: GT
        ax1 = fig.add_subplot(n, NCOLS, row * NCOLS + 1, projection='3d')
        if gt_mesh:
            render_mesh_on_ax(gt_mesh, ax1, color='cornflowerblue', alpha=0.5,
                              label=f'GT — {key}', auto_lim=True)
        else:
            ax1.set_title(f'GT missing — {key}', fontsize=9)

        # Col 2: TripoSR
        ax2 = fig.add_subplot(n, NCOLS, row * NCOLS + 2, projection='3d')
        if tri_mesh:
            render_mesh_on_ax(tri_mesh, ax2, color='salmon', alpha=0.5,
                              label=f'TripoSR — {key}', auto_lim=True)
        else:
            ax2.set_title(f'TripoSR missing — {key}', fontsize=9)

        # Col 3: Raw overlay (different scales)
        ax3 = fig.add_subplot(n, NCOLS, row * NCOLS + 3, projection='3d')
        if gt_mesh:
            render_mesh_on_ax(gt_mesh, ax3, color='cornflowerblue', alpha=0.25)
        if tri_mesh:
            render_mesh_on_ax(tri_mesh, ax3, color='salmon', alpha=0.25)
        if gt_mesh and tri_mesh:
            set_shared_lim(ax3, [gt_mesh, tri_mesh])
        ax3.set_title(f'Raw overlay — {key}', fontsize=9)

        # Col 4: ICP-aligned overlay (scale + rotation + translation)
        ax4 = fig.add_subplot(n, NCOLS, row * NCOLS + 4, projection='3d')
        if gt_mesh and tri_mesh:
            gt_aligned, sc, cost = align_gt_to_triposr(gt_mesh, tri_mesh)
            render_mesh_on_ax(gt_aligned, ax4, color='cornflowerblue', alpha=0.25)
            render_mesh_on_ax(tri_mesh, ax4, color='salmon', alpha=0.25)
            set_shared_lim(ax4, [gt_aligned, tri_mesh])
            ax4.set_title(f'ICP (s={sc:.3f} c={cost:.4f}) — {key}', fontsize=9)
        else:
            ax4.set_title(f'Aligned N/A — {key}', fontsize=9)

    plt.tight_layout()
    plt.show()

/home/markiv/sdfer/.venv/lib/python3.12/site-packages/trimesh/triangles.py:659: RuntimeWarning: invalid value encountered in divide
  v = (d1[is_ab] / (d1[is_ab] - d3[is_ab])).reshape((-1, 1))


KeyboardInterrupt: 

In [13]:
! pip install rtree

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.6/507.6 kB 27.4 MB/s eta 0:00:00
